In [ ]:
# Import required packages
from meijer import Meijer
import pandas as pd
from datetime import datetime

# Initialize the Meijer client
print("🚀 Initializing Meijer client...")
client = Meijer()

# Check authentication status
if client.auth_status.name == "AUTHENTICATED":
    print("✅ Successfully authenticated!")
else:
    print("❌ Authentication failed. Please check your credentials.")
    print("   Ensure you have auth.txt or ~/.config/meijer.txt configured")


In [ ]:
# Fetch coupons from Meijer API
print("🎫 Fetching coupons...")
coupons = client.get_coupons(limit=30)

print(f"Found {len(coupons)} coupons")

# Display basic coupon information
if coupons:
    # Count by status
    clipped_count = len([c for c in coupons if c.is_clipped])
    available_count = len([c for c in coupons if not c.is_clipped])
    expired_count = len([c for c in coupons if c.is_expired])
    
    print(f"📊 Coupon Status:")
    print(f"   🔗 {clipped_count} already clipped")
    print(f"   ⭕ {available_count} available to clip")
    print(f"   ⏰ {expired_count} expired")
else:
    print("No coupons found")


In [ ]:
# Create a DataFrame for better visualization
if coupons:
    coupon_data = []
    
    for coupon in coupons[:15]:  # Show first 15 coupons
        coupon_data.append({
            'ID': coupon.meijer_offer_id,
            'Title': coupon.title[:50],  # Truncate long titles
            'Discount': coupon.formatted_discount,
            'Status': '🔗 Clipped' if coupon.is_clipped else '⭕ Available',
            'Expired': '⏰ Yes' if coupon.is_expired else '✅ No',
            'Expires': coupon.redemption_end_date.strftime('%Y-%m-%d') if coupon.redemption_end_date else 'N/A'
        })
    
    df = pd.DataFrame(coupon_data)
    print("📋 Coupon Details (First 15):")
    display(df)
else:
    print("No coupon data to display")


In [ ]:
# Find available coupons (not clipped and not expired)
available_coupons = [c for c in coupons if not c.is_clipped and not c.is_expired]

print(f"📌 Found {len(available_coupons)} available coupons to clip")

if available_coupons:
    # Let's clip the first 3 available coupons
    coupons_to_clip = available_coupons[:3]
    
    print(f"\nClipping {len(coupons_to_clip)} coupons:")
    
    clip_results = []
    
    for i, coupon in enumerate(coupons_to_clip, 1):
        print(f"\n{i}. Clipping: {coupon.title[:40]}...")
        print(f"   Discount: {coupon.formatted_discount}")
        
        try:
            success = coupon.clip()
            if success:
                print(f"   ✅ Successfully clipped!")
                clip_results.append({'Coupon': coupon.title[:40], 'Result': '✅ Success'})
            else:
                print(f"   ❌ Failed to clip")
                clip_results.append({'Coupon': coupon.title[:40], 'Result': '❌ Failed'})
        except Exception as e:
            print(f"   ❌ Error: {e}")
            clip_results.append({'Coupon': coupon.title[:40], 'Result': f'❌ Error: {e}'})
    
    # Show results summary
    if clip_results:
        print("\n📊 Clipping Results:")
        results_df = pd.DataFrame(clip_results)
        display(results_df)
else:
    print("No available coupons to clip")


In [ ]:
# Get updated coupon list to see current status
print("🔄 Refreshing coupon list...")
updated_coupons = client.get_coupons(limit=30)

# Find clipped coupons to potentially unclip
clipped_coupons = [c for c in updated_coupons if c.is_clipped]

print(f"🔓 Found {len(clipped_coupons)} clipped coupons")

if clipped_coupons:
    # Let's unclip 2 coupons for demonstration
    coupons_to_unclip = clipped_coupons[:2]
    
    print(f"\nUnclipping {len(coupons_to_unclip)} coupons:")
    
    unclip_results = []
    
    for i, coupon in enumerate(coupons_to_unclip, 1):
        print(f"\n{i}. Unclipping: {coupon.title[:40]}...")
        print(f"   Discount: {coupon.formatted_discount}")
        
        try:
            success = coupon.unclip()
            if success:
                print(f"   ✅ Successfully unclipped!")
                unclip_results.append({'Coupon': coupon.title[:40], 'Result': '✅ Success'})
            else:
                print(f"   ❌ Failed to unclip")
                unclip_results.append({'Coupon': coupon.title[:40], 'Result': '❌ Failed'})
        except Exception as e:
            print(f"   ❌ Error: {e}")
            unclip_results.append({'Coupon': coupon.title[:40], 'Result': f'❌ Error: {e}'})
    
    # Show results summary
    if unclip_results:
        print("\n📊 Unclipping Results:")
        results_df = pd.DataFrame(unclip_results)
        display(results_df)
else:
    print("No clipped coupons to unclip")


In [ ]:
# Get final coupon status
print("📊 Final Coupon Analysis")
print("=" * 40)

final_coupons = client.get_coupons(limit=50)

if final_coupons:
    # Calculate statistics
    total = len(final_coupons)
    clipped = len([c for c in final_coupons if c.is_clipped])
    available = len([c for c in final_coupons if not c.is_clipped and not c.is_expired])
    expired = len([c for c in final_coupons if c.is_expired])
    
    # Calculate potential savings
    total_savings = 0
    clipped_savings = 0
    
    for coupon in final_coupons:
        if coupon.redeem_amount:
            total_savings += coupon.redeem_amount
            if coupon.is_clipped:
                clipped_savings += coupon.redeem_amount
    
    # Display summary statistics
    summary_data = {
        'Metric': ['Total Coupons', 'Clipped Coupons', 'Available Coupons', 'Expired Coupons', 
                   'Potential Total Savings', 'Current Clipped Savings'],
        'Value': [total, clipped, available, expired, f'${total_savings:.2f}', f'${clipped_savings:.2f}']
    }
    
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)
    
    print(f"\n💡 Key Insights:")
    print(f"• You have {clipped} coupons ready for savings")
    print(f"• {available} more coupons are available to clip")
    print(f"• Current active savings potential: ${clipped_savings:.2f}")
    if available > 0:
        print(f"• Consider clipping more coupons for additional savings!")
else:
    print("No coupon data available for analysis")
